In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print("All packages loaded!")

In [ ]:
fear_greed = pd.read_csv('fear_greed_index.csv')
historical = pd.read_csv('historical_data.csv')
print(f"Fear/Greed: {fear_greed.shape}")
print(f"Historical: {historical.shape}")
print(f"Traders: {historical['Account'].nunique()}")

In [ ]:
# Data preparation
fear_greed['date'] = pd.to_datetime(fear_greed['date'])
fear_greed['sentiment_binary'] = fear_greed['classification'].map({
    'Extreme Fear': 'Fear', 'Fear': 'Fear', 'Neutral': 'Neutral',
    'Greed': 'Greed', 'Extreme Greed': 'Greed'})

historical['date'] = pd.to_datetime(pd.to_datetime(historical['Timestamp'], unit='ms').dt.date)

trades = historical[historical['Direction'].isin(
    ['Open Long','Close Long','Open Short','Close Short','Buy','Sell'])].copy()
trades['is_long'] = trades['Direction'].isin(['Open Long','Close Long','Buy'])
trades['is_profitable'] = trades['Closed PnL'] > 0
trades['is_loss'] = trades['Closed PnL'] < 0

daily = trades.groupby(['Account','date']).agg(
    daily_pnl=('Closed PnL','sum'), daily_trades=('Closed PnL','count'),
    winning_trades=('is_profitable','sum'), losing_trades=('is_loss','sum'),
    avg_trade_size_usd=('Size USD','mean'), total_volume_usd=('Size USD','sum'),
    long_trades=('is_long','sum'), total_fees=('Fee','sum')).reset_index()

daily['win_rate'] = daily['winning_trades'] / (daily['winning_trades'] + daily['losing_trades']).fillna(0)
daily['long_ratio'] = daily['long_trades'] / daily['daily_trades']
daily['is_profitable_day'] = daily['daily_pnl'] > 0

analysis_df = daily.merge(
    fear_greed[['date','value','classification','sentiment_binary']], 
    on='date', how='inner')
print(f"Final dataset: {len(analysis_df)} rows, {analysis_df['Account'].nunique()} traders")

In [ ]:
# Fear vs Greed comparison
binary_df = analysis_df[analysis_df['sentiment_binary'].isin(['Fear','Greed'])]
fear = binary_df[binary_df['sentiment_binary']=='Fear']
greed = binary_df[binary_df['sentiment_binary']=='Greed']

perf = binary_df.groupby('sentiment_binary').agg(
    avg_pnl=('daily_pnl','mean'), avg_wr=('win_rate','mean'), 
    avg_trades=('daily_trades','mean'), avg_size=('avg_trade_size_usd','mean'),
    avg_vol=('total_volume_usd','mean'), avg_fees=('total_fees','mean'),
    avg_long=('long_ratio','mean'), prof_pct=('is_profitable_day','mean')).round(2)
print(perf)

print("\nStatistical tests:")
print(f"PnL p-value: {stats.ttest_ind(fear['daily_pnl'], greed['daily_pnl'])[1]:.4f}")
print(f"Win Rate p-value: {stats.ttest_ind(fear['win_rate'], greed['win_rate'])[1]:.4f}")
print(f"Trades p-value: {stats.ttest_ind(fear['daily_trades'], greed['daily_trades'])[1]:.4f}")

In [ ]:
# Trader segmentation
profiles = analysis_df.groupby('Account').agg(
    avg_pnl=('daily_pnl','mean'), avg_wr=('win_rate','mean'), 
    avg_trades=('daily_trades','mean'), avg_size=('avg_trade_size_usd','mean'),
    avg_long=('long_ratio','mean'), days=('date','nunique'), 
    prof_days=('is_profitable_day','sum')).reset_index()
profiles['prof_pct'] = profiles['prof_days'] / profiles['days']

med = profiles['avg_trades'].median()
profiles['freq_seg'] = profiles['avg_trades'].apply(lambda x: 'High' if x >= med else 'Low')

print(profiles.groupby('freq_seg').agg(
    N=('Account','count'), PnL=('avg_pnl','mean'), 
    WR=('avg_wr','mean'), Trades=('avg_trades','mean')).round(2))

In [ ]:
# K-Means clustering
feat = ['avg_pnl','avg_wr','avg_trades','avg_size','avg_long','prof_pct']
X = StandardScaler().fit_transform(profiles[feat].fillna(0))
profiles['cluster'] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X)
pca = PCA(2).fit_transform(X)
profiles['pca1'], profiles['pca2'] = pca[:,0], pca[:,1]
names = {0:'Balanced',1:'Long-Biased Winners',2:'High-Freq Elite',3:'Short-Biased Winners'}
profiles['arch'] = profiles['cluster'].map(names)
print(profiles.groupby('arch')[feat].mean().round(2))

In [ ]:
# Predictive model
md = analysis_df[analysis_df['sentiment_binary'].isin(['Fear','Greed'])].copy()
md['bucket'] = pd.cut(md['daily_pnl'], bins=[-float('inf'),0,50000,float('inf')], 
                      labels=['Loss','Small','Big'])
m = md[['value','daily_trades','avg_trade_size_usd','long_ratio','total_fees',
        'bucket','sentiment_binary']].dropna()
m['se'] = LabelEncoder().fit_transform(m['sentiment_binary'])
X = m[['value','daily_trades','avg_trade_size_usd','long_ratio','total_fees','se']]
y = m['bucket']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
rf = RandomForestClassifier(100, random_state=42, max_depth=5).fit(Xtr, ytr)
print(f"Accuracy: {accuracy_score(yte, rf.predict(Xte)):.4f}")
print(pd.DataFrame({'feature':X.columns,'imp':rf.feature_importances_})
      .sort_values('imp',ascending=False))